# 01 - Datenbestand

Der lokale TLFS23-Rohdatenbestand wird vor Qualitätsprüfung, Bereinigung und Datenaufteilung inventarisiert. Die Rohdaten bleiben unverändert. Ordner werden zu diesem Zeitpunkt noch nicht als verifizierte Zeichenlabels interpretiert.

**Quelle:** [TLFS23, Mendeley Data, Version 2](https://data.mendeley.com/datasets/39kzs5pxmk/2), DOI: `10.17632/39kzs5pxmk.2`, Lizenz: CC BY 4.0. Die Datensatzbeschreibung nennt 248 Klassen und 255.155 Bilder. Diese Angaben werden mit der lokalen Kopie verglichen.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "notebooks").is_dir() and (p / "data").is_dir()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projektordner mit notebooks/ und data/ nicht gefunden.")

DATASET_ROOT = PROJECT_ROOT / "data/raw/TLFS23 - Tamil Language Finger Spelling Image Dataset 2"
CLASS_ROOT = DATASET_ROOT / "Dataset Folders"
MANIFEST_DIR = PROJECT_ROOT / "data/manifests"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts/data_audit"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

if not CLASS_ROOT.is_dir():
    raise FileNotFoundError(f"Klassenordner nicht gefunden: {CLASS_ROOT}")

print("Projekt:", PROJECT_ROOT)
print("Daten:", CLASS_ROOT)

Projekt: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops
Daten: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/data/raw/TLFS23 - Tamil Language Finger Spelling Image Dataset 2/Dataset Folders


## 1. Dateien erfassen

Für jede Bilddatei werden relativer Pfad, Quellordner, Dateiendung und Dateigröße erfasst. Die Bildinhalte werden noch nicht geöffnet. Versteckte Dateien werden ignoriert. unerwartete Unterordner führen zum Abbruch.

In [2]:
class_dirs = sorted(
    (p for p in CLASS_ROOT.iterdir() if p.is_dir() and not p.name.startswith(".")),
    key=lambda p: (0, int(p.name)) if p.name.isdigit() else (1, p.name)
)
if not class_dirs:
    raise ValueError("Keine Klassenordner vorhanden.")

if any(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
       for p in CLASS_ROOT.iterdir()):
    raise ValueError("Bilddateien liegen außerhalb der Klassenordner. Struktur prüfen.")

records, other_files = [], []
for folder in class_dirs:
    for path in sorted(folder.iterdir()):
        if path.name.startswith("."):
            continue
        if path.is_dir():
            raise ValueError(f"Unerwarteter Unterordner: {path}")
        relative_path = path.relative_to(CLASS_ROOT).as_posix()
        if path.suffix.lower() not in IMAGE_EXTENSIONS:
            other_files.append(relative_path)
            continue
        records.append({
            "relative_path": relative_path,
            "dataset_folder": folder.name,
            "filename": path.name,
            "extension": path.suffix.lower(),
            "file_size_bytes": path.stat().st_size,
        })

file_inventory = pd.DataFrame(records)
del records
if file_inventory.empty:
    raise ValueError("Keine Dateien mit den vorgesehenen Bildendungen gefunden.")

print("Erfasste Bilddateien:", len(file_inventory))
print("Sonstige Dateien:", len(other_files))
display(file_inventory.head())

Erfasste Bilddateien: 255158
Sonstige Dateien: 0


,relative_path,dataset_folder,filename,extension,file_size_bytes
0,1/img_001.jpg,1,img_001.jpg,.jpg,38033
1,1/img_002.jpg,1,img_002.jpg,.jpg,71541
2,1/img_003.jpg,1,img_003.jpg,.jpg,34326
3,1/img_004.jpg,1,img_004.jpg,.jpg,30838
4,1/img_005.jpg,1,img_005.jpg,.jpg,38423


## 2. Klassenbestand zusammenfassen

Die Anzahl der Bilder wird pro Quellordner zusammengefasst. Leere Ordner bleiben sichtbar. Ordnernamen werden noch nicht als verifizierte Zeichenlabels behandelt.

In [ ]:
inventory = (
    file_inventory.groupby("dataset_folder", sort=False)
    .agg(
        n_images=("relative_path", "size"),
        size_bytes=("file_size_bytes", "sum"),
        extensions=("extension", lambda s: ", ".join(sorted(s.unique()))),
    )
    .reindex([p.name for p in class_dirs])
    .rename_axis("dataset_folder").reset_index()
)
inventory[["n_images", "size_bytes"]] = (
    inventory[["n_images", "size_bytes"]].fillna(0).astype("int64")
)
inventory[["extensions"]] = inventory[["extensions"]].fillna("")
inventory["size_mib"] = (inventory["size_bytes"] / 1024**2).round(2)

display(inventory[["dataset_folder", "n_images", "extensions", "size_mib"]].head(12))

,dataset_folder,n_images,extensions,size_mib
0,1,1064,.jpg,40.15
1,2,1062,.jpg,39.88
2,3,1061,.jpg,40.14
3,4,1056,.jpg,39.92
4,5,1048,.jpg,39.64
5,6,1025,.jpg,38.51
6,7,1078,.jpg,40.71
7,8,1042,.jpg,39.23
8,9,1059,.jpg,39.63
9,10,1055,.jpg,39.78


## 3. Umfang und Auffälligkeiten prüfen

Abweichungen von der Veröffentlichung werden dokumentiert, nicht automatisch korrigiert. Dateiendungen allein belegen weder die Lesbarkeit noch die Eindeutigkeit eines Bildes.

In [4]:
REPORTED_CLASSES = 248
REPORTED_IMAGES = 255155

comparison = pd.DataFrame({
    "Kennzahl": ["Klassenordner", "Bilddateien"],
    "Veröffentlicht": [REPORTED_CLASSES, REPORTED_IMAGES],
    "Lokal": [len(inventory), len(file_inventory)],
})
comparison["Abweichung"] = comparison["Lokal"] - comparison["Veröffentlicht"]
display(comparison)

display(inventory["n_images"].describe().to_frame("Bilder pro Klassenordner"))
print("Leere Klassenordner:", inventory.loc[inventory["n_images"].eq(0), "dataset_folder"].tolist())
print("Dateien mit 0 Byte:", int(file_inventory["file_size_bytes"].eq(0).sum()))
print("Dateigröße gesamt (GiB):", round(file_inventory["file_size_bytes"].sum() / 1024**3, 2))

if comparison["Abweichung"].ne(0).any():
    print("Abweichung festgestellt. Für die weitere Analyse gilt der erfasste lokale Bestand.")
if other_files:
    print("Beispiele sonstiger Dateien:", other_files[:5])

,Kennzahl,Veröffentlicht,Lokal,Abweichung
0,Klassenordner,248,248,0
1,Bilddateien,255155,255158,3


,Bilder pro Klassenordner
count,248.000000
mean,1028.862903
std,21.231836
min,1000.000000
25%,1011.000000
50%,1023.000000
75%,1046.250000
max,1081.000000


Leere Klassenordner: []
Dateien mit 0 Byte: 0
Dateigröße gesamt (GiB): 9.38
Abweichung festgestellt. Für die weitere Analyse gilt der erfasste lokale Bestand.


## 4. Ergebnisse speichern

Die Inventartabellen und Herkunftsinformationen werden als reproduzierbare Ausgangsbasis für die nachfolgenden Datenprüfungen gespeichert. Die Rohdaten werden nicht verändert.

In [ ]:
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

file_inventory.to_csv(MANIFEST_DIR / "file_inventory.csv", index=False)
inventory.to_csv(MANIFEST_DIR / "class_inventory.csv", index=False)
comparison.to_csv(ARTIFACT_DIR / "01_source_comparison.csv", index=False)

summary = {
    "dataset_name": "TLFS23", "dataset_version": 2,
    "source": "Mendeley Data", "doi": "10.17632/39kzs5pxmk.2",
    "source_url": "https://data.mendeley.com/datasets/39kzs5pxmk/2",
    "license": "CC BY 4.0",
    "dataset_root": str(DATASET_ROOT), "class_root": str(CLASS_ROOT),
    "class_root_relative": CLASS_ROOT.relative_to(PROJECT_ROOT).as_posix(),
    "reported_n_class_folders": REPORTED_CLASSES, "reported_n_images": REPORTED_IMAGES,
    "n_class_folders": len(inventory), "n_images": len(file_inventory),
    "image_extensions": sorted(IMAGE_EXTENSIONS),
    "other_files": other_files,
    "raw_data_modified": False,
}
(ARTIFACT_DIR / "01_inventory_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Inventare:", MANIFEST_DIR)
print("Zusammenfassung und Abbildung:", ARTIFACT_DIR)

Inventare: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/data/manifests
Zusammenfassung und Abbildung: /Users/sharu/Documents/DBU/tamil-finger-sign-mlops/artifacts/data_audit


## Ergebnis

Der lokale Datenbestand umfasst 248 Klassenordner mit 255.158 Bilddateien und 9,38 GiB. Gegenüber der Datensatzbeschreibung wurden drei zusätzliche Bilddateien festgestellt. Leere Klassenordner oder Dateien mit 0 Byte wurden nicht gefunden. Für die weiteren Verarbeitungsschritte gilt der inventarisierte lokale Bestand als Referenz.